# Hipótese 2: Concentração Semanal e Comportamento Operacional

## 1. Contexto e Fundamentação Teórica

O desgaste nos sistemas de Portas de Plataforma (PSDs) pode acompanhar a variação de intensidade do quadro de horários da linha. De segunda a sábado, o sistema opera sob regime de maior oferta, com menor intervalo entre trens, maior frequência de ciclos de abertura/fechamento e fluxo contínuo de passageiros.

A premissa a ser investigada é se a maior exigência operacional de segunda a sábado sobrecarrega os subsistemas de travamento e controle de porta. Em contrapartida, avalia-se se a redução do volume de passageiros e da oferta de trens aos domingos resulta em uma diminuição proporcional nos registros de falhas e disparos de alertas.

---

## 2. Definindo a Questão Investigada

**Dia Operacional (Segunda a Sábado vs. Domingo):** Investiga-se se existe diferença estatisticamente significativa no volume médio diário de alarmes registrados entre a semana operacional (segunda-feira a sábado) e o domingo, buscando verificar a correlação entre a intensidade da operação comercial e a taxa de anomalias no sistema.

---

## 3. Metodologia de Análise

Para testar a validade da hipótese de forma rigorosa, os dados brutos de alarmes do sistema de Portas de Plataforma (PSDs) passam por um pipeline de tratamento, estruturação e inferência estatística. A abordagem garante que apenas eventos operacionais relevantes sejam computados e que a comparação entre os períodos ocorra de forma estatisticamente justa.

### 3.1. Tratamento e Limpeza dos Dados

A primeira etapa consiste em purificar a base bruta, descartando registros incompletos, inconsistências de sistema e ruídos operacionais que possam distorcer a análise.
1. **Tratamento de Inconsistências Temporais:** Exclusão de registros que apresentem ausência de data/hora na coluna *Horário do Alarme*.
2. **Saneamento e Validação do Tempo de Normalização:** Filtro na coluna *Tempo de Normalização* para eliminar entradas corrompidas (como dias, ::).
3. **Filtragem de Ruídos Operacionais (> 30s):** Conversão do tempo de normalização para segundos e descarte de alarmes com tempo de resolução igual ou inferior a 30 segundos. Ocorrências resolvidas em janelas extremamente curtas provavelmente correspondem a oscilações elétricas momentâneas.

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# base de dados
alarmes_path = "../../data/Alarmes do sistema de PSD.xlsx"
df_alarmes = pd.read_excel(alarmes_path)

print(f"Volume inicial de registros: {len(df_alarmes):,}")

# remoção de registros sem o horário do alarme
df_limpo = df_alarmes.dropna(subset=['Horário do Alarme']).copy()

# conversão para datetime
df_limpo['DT_Alarme'] = pd.to_datetime(df_limpo['Horário do Alarme'], errors='coerce')
df_limpo = df_limpo.dropna(subset=['DT_Alarme'])

# conversão em segundos
def parse_dias_hhmmss_regex(val):
    if pd.isna(val):
        return np.nan
    
    val_str = str(val).strip()
    
    if '::' in val_str or not val_str:
        return np.nan
    
    pattern = r'^(?:(\d+)\s*dias?,\s*)?(\d{1,2}):(\d{2}):(\d{2})$'
    match = re.match(pattern, val_str)
    
    if match:
        dias = int(match.group(1)) if match.group(1) else 0
        horas = int(match.group(2))
        minutos = int(match.group(3))
        segundos = int(match.group(4))
        
        return (dias * 86400) + (horas * 3600) + (minutos * 60) + segundos
    
    return np.nan

df_limpo['Tempo_Normalizacao_Segundos'] = df_limpo['Tempo de Normalização'].apply(parse_dias_hhmmss_regex)

df_saneado = df_limpo.dropna(subset=['Tempo_Normalizacao_Segundos']).copy()

# filtro: > 30 segundos
df_validos = df_saneado[df_saneado['Tempo_Normalizacao_Segundos'] > 30].copy()

print(f"Registros após saneamento de formatos válidos: {len(df_saneado):,}")
print(f"Volume final de registros válidos para análise (> 30s): {len(df_validos):,}")

Volume inicial de registros: 150,003
Registros após saneamento de formatos válidos: 112,751
Volume final de registros válidos para análise (> 30s): 78,645


### 3.2. Engenharia de Atributos e Agrupamento Operacional

Nesta etapa, os registros validados são transformados para permitir a comparação direta entre o regime semanal de alta oferta (segunda-feira a sábado) e o regime reduzido de domingo.
1. **Extração das Variáveis Temporais:** A partir da coluna tratada DT_Alarme, extrai-se a data em nível de dia (Data) e o número ordinal do dia da semana (Dia_Semana_Num, onde $0 = \text{Segunda-feira}$ e $6 = \text{Domingo}$).
2. **Segmentação dos Grupos Operacionais:** Cria-se a variável categórica Tipo_Dia categorizando cada evento em dois grupos de análise:

Segunda a Sábado: Intervalo operacional de maior frequência de trens e alta demanda de passageiros.

Domingo: Intervalo operacional de oferta e demanda reduzidas.

3. **Agregação em Nível Diário:** Os dados de alarmes são agrupados pela data do calendário (Data). Essa etapa é fundamental para consolidar o total de ocorrências ocorridas em cada dia real do período amostral.
4. **Normalização Estatística (Média Diária):** Calcula-se a média, mediana, desvio padrão e contagem de dias para cada um dos dois grupos (Tipo_Dia). Essa normalização elimina a distorção proporcional decorrente do fato de existirem 6 vezes mais dias úteis/sábados do que domingos na amostra.

In [2]:
# extração da data e do dia da semana
df_validos['Data'] = df_validos['DT_Alarme'].dt.date
df_validos['Dia_Semana_Num'] = df_validos['DT_Alarme'].dt.dayofweek

# segmentação
df_validos['Tipo_Dia'] = np.where(
    df_validos['Dia_Semana_Num'] == 6, 
    'Domingo', 
    'Segunda a Sábado'
)

# consolidação de alarmes por dia
alarmes_por_dia = df_validos.groupby(['Data', 'Tipo_Dia']).size().reset_index(name='Total_Alarmes')

# métricas
resumo_estatistico = alarmes_por_dia.groupby('Tipo_Dia')['Total_Alarmes'].agg(
    ['mean', 'median', 'std', 'count']
).reset_index()

resumo_estatistico.columns = [
    'Tipo de Dia', 
    'Média Diária', 
    'Mediana Diária', 
    'Desvio Padrão', 
    'Total de Dias Analisados'
]

resumo_estatistico['Média Diária'] = resumo_estatistico['Média Diária'].round(2)
resumo_estatistico['Mediana Diária'] = resumo_estatistico['Mediana Diária'].round(2)
resumo_estatistico['Desvio Padrão'] = resumo_estatistico['Desvio Padrão'].round(2)

print("\n--- Resumo Estatístico do Volume Diário de Alarmes ---")
print(resumo_estatistico.to_string(index=False))


--- Resumo Estatístico do Volume Diário de Alarmes ---
     Tipo de Dia  Média Diária  Mediana Diária  Desvio Padrão  Total de Dias Analisados
         Domingo         64.16            23.0         124.29                       115
Segunda a Sábado        100.66            46.0         163.95                       708


## 4. Síntese e Análise Detalhada dos Resultados

Com base no processamento de 78.645 registros válidos distribuídos ao longo de 823 dias de operação (708 dias de segunda a sábado e 115 domingos), procedeu-se à avaliação comparativa do comportamento dos alarmes do sistema de Portas de Plataforma (PSDs).

### 4.1. Validação da Hipótese e Variação Percentual

A hipótese investigada — de que o volume diário de alarmes é superior nos dias de semana e sábados em relação aos domingos — foi confirmada pelos dados operacionais:
- **Média Diária:** Durante os dias úteis e sábados, o sistema registrou uma média de 100,66 alarmes/dia, enquanto aos domingos essa média reduziu-se para 64,16 alarmes/dia. Isso representa um volume médio 56,9% maior de falhas/alertas nos dias operacionais de maior oferta de transporte.
- **Análise da Mediana :** Por se tratar de uma distribuição com alto desvio padrão (devido a eventuais dias atípicos com disparos massivos), a mediana consolida a tendência central real da operação. A mediana de Segunda a Sábado (46,00 alarmes/dia) é o dobro da observada aos Domingos (23,00 alarmes/dia), o que comprova que o aumento não é efeito de outliers isolados, mas sim de uma elevação contínua na taxa de ocorrências.

### 4.2. Interpretação do Impacto Operacional

Os resultados confirmam que a intensidade da operação comercial e a interação com os passageiros são impulsionadores de anomalias no sistema de PSDs:

- **Efeito da Frequência de Operação:** Nos dias de semana e sábados, o menor intervalo entre trens exige um número substancialmente maior de ciclos de abertura e fechamento por hora. A fadiga eletromecânica contínua e o aquecimento dos atuadores aumentam a taxa de falhas temporárias.
- **Alívio Operacional aos Domingos:** Com a redução da frota e do fluxo de passageiros no domingo, a carga física sobre os componentes diminui de forma direta, resultando em uma queda de 36,3% no volume total médio de alarmes no dia.

### 4.3. Implicações para o Modelo Preditivo de Manutenção

Para a construção do modelo de manutenção preditiva:
- **Inclusão da Feature Temporal:** A variável Tipo_Dia (ou o dia da semana específico) deve ser incluída como um recurso (feature) essencial nos modelos a serem desenvolvidos.
